# Testes de Scraping em Fontes Web

Avaliação da viabilidade de extração de dados das fontes indicadas para possível utilização na plataforma de relatórios agênticos do SenacLabs.

# Objetivo

Avaliar a viabilidade de scraping das fontes web indicadas, verificando:

- disponibilidade e resposta HTTP;
- estrutura do conteúdo retornado;
- possibilidade de extração de dados utilizando Python;
- possíveis mecanismos de bloqueio ou anti-bot;
- possível dependência de JavaScript para carregamento do conteúdo;
- comportamento e estabilidade básica das fontes durante os testes.

O Fluxo de dados extraidos será voltado para a plataforma de criação de relatórios agênticos do SenacLabs


## Dependências

Para os testes serão utilizadas bibliotecas populares do ecossistema Python:

- `requests`: realização de requisições HTTP;
- `BeautifulSoup`: leitura e análise do HTML;
- `dataclasses`: estruturação dos resultados;
- `urllib.parse`: manipulação das URLs;
- `time`: medição do tempo de resposta.

In [49]:
from dataclasses import dataclass
from urllib.parse import urlparse
import time

import requests
from bs4 import BeautifulSoup

## Estruturas auxiliares

Os resultados de cada teste serão armazenados em uma estrutura padronizada para facilitar a comparação entre as fontes.

In [50]:
@dataclass
class TestResult:
    url: str
    status_code: int | None
    response_time: float | None
    content_length: int | None
    content_type: str | None
    title: str | None
    robots_status: int | None
    possible_block: bool
    block_indicators: list[str]
    possible_js_required: bool
    error: str | None

A classe abaixo concentra os testes básicos realizados em cada fonte.

In [51]:
class SourceTester:
    def __init__(self):
        self.session = requests.Session()

        self.session.headers.update({
            "User-Agent": "SourceCompatibilityTester/1.0"
        })

    def test(self, url: str) -> TestResult:
        print(f"\n{'=' * 60}")
        print(f"Testando: {url}")
        print(f"{'=' * 60}")

        robots_status = self._check_robots(url)

        try:
            start = time.perf_counter()

            response = self.session.get(
                url,
                timeout=15,
                allow_redirects=True
            )

            elapsed = time.perf_counter() - start

            content_type = response.headers.get("Content-Type")

            print(f"Status: {response.status_code}")
            print(f"URL final: {response.url}")
            print(f"Tempo: {elapsed:.2f}s")
            print(f"Tamanho: {len(response.content)} bytes")
            print(f"Content-Type: {content_type}")

            title = self._extract_title(response.text)

            block_indicators = self._detect_block_reasons(response)

            blocked = (
                response.status_code in {401, 403, 429, 503}
                or len(block_indicators) > 0
            )

            js_required = self._detect_possible_js_page(response.text)

            print(f"Título: {title}")
            print(f"Possível bloqueio: {blocked}")
            print(f"Possível dependência de JS: {js_required}")

            return TestResult(
                url=url,
                status_code=response.status_code,
                response_time=elapsed,
                content_length=len(response.content),
                content_type=content_type,
                title=title,
                robots_status=robots_status,
                possible_block=blocked,
                block_indicators=block_indicators,
                possible_js_required=js_required,
                error=None
            )

        except requests.RequestException as exc:
            print(f"Erro de requisição: {exc}")

            return TestResult(
                url=url,
                status_code=None,
                response_time=None,
                content_length=None,
                content_type=None,
                title=None,
                robots_status=robots_status,
                possible_block=False,
                block_indicators=[],
                possible_js_required=False,
                error=str(exc)
            )

    def _check_robots(self, url: str) -> int | None:
        parsed = urlparse(url)

        robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

        try:
            response = self.session.get(
                robots_url,
                timeout=10
            )

            print(f"robots.txt: {response.status_code}")

            return response.status_code

        except requests.RequestException:
            print("robots.txt: não foi possível consultar.")
            return None

    @staticmethod
    def _extract_title(html: str) -> str | None:
        soup = BeautifulSoup(html, "html.parser")

        if soup.title and soup.title.string:
            return soup.title.string.strip()

        return None

    @staticmethod
    def _detect_block_reasons(response: requests.Response) -> list[str]:
        text = response.text.lower()

        indicators = [
            "access denied",
            "too many requests",
            "captcha",
            "verify you are human",
            "checking your browser",
        ]

        return [
            indicator
            for indicator in indicators
            if indicator in text
        ]

    @staticmethod
    def _detect_possible_js_page(html: str) -> bool:
        soup = BeautifulSoup(html, "html.parser")

        text = soup.get_text(" ", strip=True)

        scripts = soup.find_all("script")

        return len(text) < 200 and len(scripts) >= 5


## Fontes avaliadas

As fontes abaixo foram indicadas para os testes. As três primeiras possuem prioridade.

In [52]:
sources = [
    "https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada",
    "https://www.gov.br/anvisa/pt-br",
    "https://www.gov.br/imprensanacional/pt-br",
    "https://www25.senado.leg.br/web/atividade",
    "https://www.al.rn.leg.br",
    "https://deirn.sdoe.com.br",
    "https://www.cmnat.rn.gov.br",
    "https://legislativo.camaraparnamirim.rn.gov.br/",
    "https://www.mossoro.rn.leg.br",
    "https://www.gov.br/anvisa",
    "https://www.gov.br/conitec",
    "https://www.gov.br/saude",
    "https://www.confaz.fazenda.gov.br",
    "https://www.gov.br/cade",
    "https://www.gov.br/senacon"
]

## Teste inicial de acessibilidade

Nesta etapa são verificadas características básicas das fontes, como:

- código HTTP;
- tempo de resposta;
- tamanho do conteúdo;
- tipo do conteúdo retornado;
- título da página;
- disponibilidade do `robots.txt`;
- possíveis indícios de bloqueio;
- possível dependência de JavaScript.

In [53]:
tester = SourceTester()

results = []

for source in sources:
    result = tester.test(source)
    results.append(result)

    time.sleep(2)


Testando: https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
robots.txt: 200
Status: 200
URL final: https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
Tempo: 0.13s
Tamanho: 32131 bytes
Content-Type: text/html; charset=utf-8
Título: Portal da Câmara dos Deputados - Busca - Portal da Câmara dos Deputados
Possível bloqueio: False
Possível dependência de JS: False

Testando: https://www.gov.br/anvisa/pt-br
robots.txt: 200
Status: 200
URL final: https://www.gov.br/anvisa/pt-br
Tempo: 0.31s
Tamanho: 579733 bytes
Content-Type: text/html; charset=utf-8
Título: Português (Brasil)
Possível bloqueio: True
Possível dependência de JS: False

Testando: https://www.gov.br/imprensanacional/pt-br
robots.txt: 200
Status: 200
URL final: https://www.gov.br/imprensanacional/pt-br
Tempo: 0.16s
Tamanho: 322085 bytes
Content-Type: text/html; charset=utf-8
Título: Imprensa Nacional
Possível bloqueio: True
Possível dependência de JS: False

Testando: https://www25.se

## Resultados consolidados

Os resultados obtidos nos testes iniciais são apresentados abaixo para facilitar a comparação entre as fontes

In [54]:
import pandas as pd

df_results = pd.DataFrame(
    [vars(result) for result in results]
)

df_results

,url,status_code,response_time,content_length,content_type,title,robots_status,possible_block,block_indicators,possible_js_required,error
0,https://www.camara.leg.br/busca-portal/proposi...,200,0.125545,32131,text/html; charset=utf-8,Portal da Câmara dos Deputados - Busca - Porta...,200,False,[],False,None
1,https://www.gov.br/anvisa/pt-br,200,0.314589,579733,text/html; charset=utf-8,Português (Brasil),200,True,[captcha],False,None
2,https://www.gov.br/imprensanacional/pt-br,200,0.157956,322085,text/html; charset=utf-8,Imprensa Nacional,200,True,[captcha],False,None
3,https://www25.senado.leg.br/web/atividade,200,0.233667,83672,text/html;charset=UTF-8,Home - Senado Federal,200,False,[],False,None
4,https://www.al.rn.leg.br,200,0.577915,87272,text/html; charset=UTF-8,Assembleia Legislativa do Rio Grande do Norte,200,True,[captcha],False,None
5,https://deirn.sdoe.com.br,403,0.126050,1233,text/html,403 - Forbidden: Access is denied.,404,True,[],False,None
6,https://www.cmnat.rn.gov.br,200,0.100317,73347,text/html; charset=UTF-8,Câmara Municipal de Natal,404,True,[captcha],False,None
7,https://legislativo.camaraparnamirim.rn.gov.br/,200,0.797363,198808,text/html; charset=utf-8,Câmara Municipal de Parnamirim - Portal Oficia...,200,True,[captcha],False,None
8,https://www.mossoro.rn.leg.br,200,0.311187,57351,text/html;charset=utf-8,CMM — Câmara Municipal de Mossoró,200,False,[],False,None
9,https://www.gov.br/anvisa,200,0.376164,579733,text/html; charset=utf-8,Português (Brasil),200,True,[captcha],False,None


## Investigação de possíveis bloqueios

In [55]:
def detect_block_reason(response):
    text = response.text.lower()

    indicators = [
        "access denied",
        "too many requests",
        "captcha",
        "verify you are human",
        "checking your browser",
    ]

    return [
        indicator
        for indicator in indicators
        if indicator in text
    ]

In [56]:
suspected_sources = [
    "https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada",
    "https://www.gov.br/anvisa/pt-br",
    "https://www.gov.br/imprensanacional/pt-br",
    "https://www.al.rn.leg.br",
    "https://www.cmnat.rn.gov.br",
    "https://legislativo.camaraparnamirim.rn.gov.br/",
    "https://www.gov.br/anvisa",
    "https://www.gov.br/conitec",
]

for source in suspected_sources:
    response = tester.session.get(source, timeout=15)

    print(source)
    print(detect_block_reason(response))
    print()

https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
[]

https://www.gov.br/anvisa/pt-br
['captcha']

https://www.gov.br/imprensanacional/pt-br
['captcha']

https://www.al.rn.leg.br
['captcha']

https://www.cmnat.rn.gov.br
['captcha']

https://legislativo.camaraparnamirim.rn.gov.br/
['captcha']

https://www.gov.br/anvisa
['captcha']

https://www.gov.br/conitec
['captcha']



## Teste de extração

### Câmara dos Deputados

Será realizada uma inspeção inicial do HTML retornado para verificar se existem elementos e conteúdos que possam ser extraídos diretamente.

In [57]:
url = sources[0]

response = requests.get(url, timeout=15)

soup = BeautifulSoup(response.text, "html.parser")

print("Título:", soup.title.string if soup.title else None)
print("Links encontrados:", len(soup.find_all("a")))

Título: 
      Portal da Câmara dos Deputados - Busca - Portal da Câmara dos Deputados
  
Links encontrados: 70


In [58]:
for link in soup.find_all("a")[:10]:
    text = link.get_text(" ", strip=True)

    if text:
        print(text, "->", link.get("href"))

Ir ao conteúdo -> #main-content
Ir à navegação principal -> #main-nav
Página inicial -> https://www.camara.leg.br
Acessibilidade -> https://www2.camara.leg.br/acessibilidade/recursos-de-acessibilidade
Fale Conosco -> https://www.camara.leg.br/fale-conosco
Congresso -> http://www.congressonacional.leg.br
Senado -> http://www.senado.leg.br
English EN -> https://www2.camara.leg.br/english
Español ES -> https://www2.camara.leg.br/espanol
Entrar -> https://www.camara.leg.br/login


In [59]:
from urllib.parse import urljoin


def test_extraction(url):
    print("\n" + "=" * 70)
    print("TESTE DE EXTRAÇÃO")
    print(url)
    print("=" * 70)

    try:
        response = requests.get(
            url,
            timeout=15
        )

        print("HTTP:", response.status_code)

        if response.status_code != 200:
            print("Extração não realizada devido ao status HTTP.")
            return

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title
            else None
        )


        headings = []

        for tag in soup.find_all(["h1", "h2", "h3"]):
            text = tag.get_text(" ", strip=True)

            if text:
                headings.append(text)

        paragraphs = []

        for tag in soup.find_all("p"):
            text = tag.get_text(" ", strip=True)

            if text:
                paragraphs.append(text)

        links = []

        for tag in soup.find_all("a", href=True):

            href = tag.get("href")

            if not href:
                continue

            links.append({
                "text": tag.get_text(" ", strip=True),
                "url": urljoin(response.url, href)
            })
            
        print("Título:", title)
        print("Headings:", len(headings))
        print("Parágrafos:", len(paragraphs))
        print("Links:", len(links))

        print("\nExemplos de headings:")

        for heading in headings[:5]:
            print(" -", heading)

        print("\nExemplos de links:")

        for link in links[:5]:
            print(
                f" - {link['text']!r} -> {link['url']}"
            )

    except requests.RequestException as error:
        print("Erro:", error)

In [60]:
for url in sources:
    test_extraction(url)


TESTE DE EXTRAÇÃO
https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada
HTTP: 200
Título: Portal da Câmara dos Deputados - Busca - Portal da Câmara dos Deputados
Headings: 1
Parágrafos: 2
Links: 70

Exemplos de headings:
 - Comunicar erro ou fazer sugestão

Exemplos de links:
 - 'Ir ao conteúdo' -> https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada#main-content
 - 'Ir à navegação principal' -> https://www.camara.leg.br/busca-portal/proposicoes/pesquisa-simplificada#main-nav
 - 'Página inicial' -> https://www.camara.leg.br
 - 'Acessibilidade' -> https://www2.camara.leg.br/acessibilidade/recursos-de-acessibilidade
 - 'Fale Conosco' -> https://www.camara.leg.br/fale-conosco

TESTE DE EXTRAÇÃO
https://www.gov.br/anvisa/pt-br
HTTP: 200
Título: Português (Brasil)
Headings: 13
Parágrafos: 5
Links: 111

Exemplos de headings:
 - Português (Brasil)
 - Anvisa aprova regularização de cosméticos artesanais
 - CMED divulga 9° edição do Anuário Estatístico do

## Teste de Rate limit dentro dos testes de scraping

Aqui a gente vai testar de forma automática se os sites bloqueiam o scraping. No final de tudo será gerado tanto os 150 resultados pois são 15 URLs e no final terá a opção de fazer um resumo do resultado final de cada URL.

In [61]:
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup


def repeated_test(url, attempts=10, delay=2):
    results = []

    session = requests.Session()

    for attempt in range(1, attempts + 1):

        try:
            start = time.perf_counter()

            response = session.get(
                url,
                timeout=15
            )

            elapsed = time.perf_counter() - start

            soup = BeautifulSoup(
                response.text,
                "html.parser"
            )

            text = soup.get_text(" ", strip=True)

            title = (
                soup.title.get_text(" ", strip=True)
                if soup.title
                else None
            )

            results.append({
                "attempt": attempt,
                "status": response.status_code,
                "response_time": round(elapsed, 3),
                "content_length": len(response.content),
                "text_length": len(text),
                "links": len(soup.find_all("a")),
                "headings": len(
                    soup.find_all(["h1", "h2", "h3"])
                ),
                "captcha_reference": (
                    "captcha" in response.text.lower()
                ),
                "title": title
            })

        except requests.RequestException as error:

            results.append({
                "attempt": attempt,
                "status": None,
                "response_time": None,
                "content_length": 0,
                "text_length": 0,
                "links": 0,
                "headings": 0,
                "captcha_reference": False,
                "title": None,
                "error": str(error)
            })

        if attempt < attempts:
            time.sleep(delay)

    return results

In [62]:
all_repeated_results = []

for url in sources:

    results = repeated_test(
        url,
        attempts=10,
        delay=2
    )

    for result in results:
        result["url"] = url
        all_repeated_results.append(result)

In [63]:
df_repeated = pd.DataFrame(all_repeated_results)

df_repeated

,attempt,status,response_time,content_length,text_length,links,headings,captcha_reference,title,url
0,1,200,0.447,32131,1961,70,1,False,Portal da Câmara dos Deputados - Busca - Porta...,https://www.camara.leg.br/busca-portal/proposi...
1,2,200,0.088,32131,1961,70,1,False,Portal da Câmara dos Deputados - Busca - Porta...,https://www.camara.leg.br/busca-portal/proposi...
2,3,200,0.160,32131,1961,70,1,False,Portal da Câmara dos Deputados - Busca - Porta...,https://www.camara.leg.br/busca-portal/proposi...
3,4,200,0.139,32131,1961,70,1,False,Portal da Câmara dos Deputados - Busca - Porta...,https://www.camara.leg.br/busca-portal/proposi...
4,5,200,0.140,32131,1961,70,1,False,Portal da Câmara dos Deputados - Busca - Porta...,https://www.camara.leg.br/busca-portal/proposi...
...,...,...,...,...,...,...,...,...,...,...
145,6,200,0.159,186394,14510,500,4,False,Senacon — Ministério da Justiça e Segurança Pú...,https://www.gov.br/senacon
146,7,200,0.158,186394,14510,500,4,False,Senacon — Ministério da Justiça e Segurança Pú...,https://www.gov.br/senacon
147,8,200,0.154,186394,14510,500,4,False,Senacon — Ministério da Justiça e Segurança Pú...,https://www.gov.br/senacon
148,9,200,0.159,186394,14510,500,4,False,Senacon — Ministério da Justiça e Segurança Pú...,https://www.gov.br/senacon


In [64]:
summary = (
    df_repeated
    .groupby("url")
    .agg(
        attempts=("attempt", "count"),

        status_min=("status", "min"),
        status_max=("status", "max"),

        response_time_avg=("response_time", "mean"),

        content_length_min=("content_length", "min"),
        content_length_max=("content_length", "max"),

        text_length_min=("text_length", "min"),
        text_length_max=("text_length", "max"),

        links_min=("links", "min"),
        links_max=("links", "max"),

        headings_min=("headings", "min"),
        headings_max=("headings", "max"),

        captcha_detected=("captcha_reference", "any")
        
    )
    .reset_index()
)

summary

,url,attempts,status_min,status_max,response_time_avg,content_length_min,content_length_max,text_length_min,text_length_max,links_min,links_max,headings_min,headings_max,captcha_detected
0,https://deirn.sdoe.com.br,10,403,403,0.2318,1233,1233,181,181,0,0,3,3,False
1,https://legislativo.camaraparnamirim.rn.gov.br/,10,200,200,1.0431,198808,198808,30137,30137,215,215,23,23,True
2,https://www.al.rn.leg.br,10,200,200,0.5481,87272,87272,6264,6264,230,230,11,11,True
3,https://www.camara.leg.br/busca-portal/proposi...,10,200,200,0.1556,32131,32131,1961,1961,70,70,1,1,False
4,https://www.cmnat.rn.gov.br,10,200,200,0.1566,73347,73347,6770,6770,142,142,40,40,True
5,https://www.confaz.fazenda.gov.br,10,200,200,0.2165,105979,105979,28629,28629,187,187,4,4,False
6,https://www.gov.br/anvisa,10,200,200,0.4417,581123,607204,3362,3624,112,112,13,13,True
7,https://www.gov.br/anvisa/pt-br,10,200,200,0.2538,579733,579802,3364,3364,111,111,13,13,True
8,https://www.gov.br/cade,10,200,200,0.2390,200215,200215,13857,13857,499,499,8,8,False
9,https://www.gov.br/conitec,10,200,200,0.4176,369363,369433,2370,2370,80,80,5,5,True


## Tabela etalhada do teste de scraping.

Todos os testes executados foi para analisar o HTML das páginas solicitadas. 

O fluxo foi.:
Acesso a página - Extração do HTML - Estabilidade e verificação de rate limit. Falta apenas a navegação interna que será executada nos próximos testes.

In [65]:
summary["status_changed"] = (
    summary["status_min"] != summary["status_max"]
)

summary["structure_changed"] = (
    (summary["links_min"] != summary["links_max"])
    |
    (summary["headings_min"] != summary["headings_max"])
)

summary["text_changed"] = (
    summary["text_length_min"]
    !=
    summary["text_length_max"]
)

summary["html_changed"] = (
    summary["content_length_min"]
    !=
    summary["content_length_max"]
)

summary["possible_rate_limit"] = (
    (summary["status_min"] == 429)
    |
    (summary["status_max"] == 429)
)

summary["stable_access"] = (
    (summary["status_min"] == 200)
    &
    (summary["status_max"] == 200)
)

In [66]:
summary[
    [
        "url",
        "attempts",
        "status_min",
        "status_max",
        "stable_access",
        "status_changed",
        "html_changed",
        "text_changed",
        "structure_changed",
        "possible_rate_limit",
        "captcha_detected"
    ]
]

,url,attempts,status_min,status_max,stable_access,status_changed,html_changed,text_changed,structure_changed,possible_rate_limit,captcha_detected
0,https://deirn.sdoe.com.br,10,403,403,False,False,False,False,False,False,False
1,https://legislativo.camaraparnamirim.rn.gov.br/,10,200,200,True,False,False,False,False,False,True
2,https://www.al.rn.leg.br,10,200,200,True,False,False,False,False,False,True
3,https://www.camara.leg.br/busca-portal/proposi...,10,200,200,True,False,False,False,False,False,False
4,https://www.cmnat.rn.gov.br,10,200,200,True,False,False,False,False,False,True
5,https://www.confaz.fazenda.gov.br,10,200,200,True,False,False,False,False,False,False
6,https://www.gov.br/anvisa,10,200,200,True,False,True,True,False,False,True
7,https://www.gov.br/anvisa/pt-br,10,200,200,True,False,True,False,False,False,True
8,https://www.gov.br/cade,10,200,200,True,False,False,False,False,False,False
9,https://www.gov.br/conitec,10,200,200,True,False,True,False,False,False,True


## Teste de navegação interna

O teste atual vai evitar puxar tudo q testamos anteriormente e verificar se seremos direcionados para outras URLs quando fizermos a tentativa de verificação interna ou se conseguimos extrair todos os dados que iremos precisar.

In [67]:
from urllib.parse import urljoin, urlparse


def get_internal_links(base_url, soup):
    internal_links = []

    base_domain = urlparse(base_url).netloc

    for tag in soup.find_all("a", href=True):
        href = tag.get("href", "").strip()

        if not href:
            continue

        if href.startswith("#"):
            continue

        if href.startswith("javascript:"):
            continue

        if href.startswith("mailto:"):
            continue

        if href.startswith("tel:"):
            continue

        absolute_url = urljoin(base_url, href)

        parsed = urlparse(absolute_url)

        if parsed.scheme not in ("http", "https"):
            continue

        if parsed.netloc != base_domain:
            continue

        clean_url = parsed._replace(fragment="").geturl()

        if clean_url not in internal_links:
            internal_links.append(clean_url)

    return internal_links

In [68]:
import time
import requests
from bs4 import BeautifulSoup


def test_internal_page(url, timeout=15):

    try:
        start = time.perf_counter()

        response = requests.get(
            url,
            timeout=timeout
        )

        elapsed = time.perf_counter() - start

        soup = BeautifulSoup(
            response.text,
            "html.parser"
        )

        text = soup.get_text(
            " ",
            strip=True
        )

        title = (
            soup.title.get_text(" ", strip=True)
            if soup.title
            else None
        )

        return {
            "internal_url": url,

            "status": response.status_code,

            "response_time": round(
                elapsed,
                3
            ),

            "content_length": len(
                response.content
            ),

            "text_length": len(text),

            "links": len(
                soup.find_all("a")
            ),

            "headings": len(
                soup.find_all(
                    ["h1", "h2", "h3"]
                )
            ),

            "captcha_reference": (
                "captcha"
                in response.text.lower()
            ),

            "title": title,

            "error": None
        }

    except requests.RequestException as error:

        return {
            "internal_url": url,
            "status": None,
            "response_time": None,
            "content_length": 0,
            "text_length": 0,
            "links": 0,
            "headings": 0,
            "captcha_reference": False,
            "title": None,
            "error": str(error)
        }

In [69]:
def test_internal_navigation(
    source_url,
    max_pages=3,
    delay=1
):

    results = []

    try:
        response = requests.get(
            source_url,
            timeout=15
        )

    except requests.RequestException as error:
        return [{
            "source_url": source_url,
            "internal_url": None,
            "status": None,
            "error": str(error)
        }]

    if response.status_code != 200:
        return [{
            "source_url": source_url,
            "internal_url": None,
            "status": response.status_code,
            "error": "Página inicial não acessível"
        }]

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    internal_links = get_internal_links(
        response.url,
        soup
    )

    internal_links = [
        link
        for link in internal_links
        if link.rstrip("/")
        != response.url.rstrip("/")
    ]

    selected_links = internal_links[:max_pages]

    for link in selected_links:

        result = test_internal_page(link)

        result["source_url"] = source_url

        results.append(result)

        time.sleep(delay)

    if not results:
        results.append({
            "source_url": source_url,
            "internal_url": None,
            "status": None,
            "error": "Nenhum link interno encontrado"
        })

    return results

In [70]:
navigation_results = []

for source_url in sources:

    results = test_internal_navigation(
        source_url,
        max_pages=3,
        delay=1
    )

    navigation_results.extend(results)

In [71]:
df_navigation = pd.DataFrame(
    navigation_results
)

df_navigation

,internal_url,status,response_time,content_length,text_length,links,headings,captcha_reference,title,error,source_url
0,https://www.camara.leg.br,200.0,0.591,88572.0,4457.0,108.0,13.0,False,Portal da Câmara dos Deputados,NaN,https://www.camara.leg.br/busca-portal/proposi...
1,https://www.camara.leg.br/fale-conosco,200.0,0.533,47849.0,4181.0,84.0,13.0,False,Fale Conosco - Portal da Câmara dos Deputados,NaN,https://www.camara.leg.br/busca-portal/proposi...
2,https://www.camara.leg.br/login,200.0,1.286,4464.0,249.0,6.0,3.0,False,Câmara dos Deputados - Entre para continuar,NaN,https://www.camara.leg.br/busca-portal/proposi...
3,https://www.gov.br/anvisa/,200.0,0.683,581179.0,3362.0,112.0,13.0,True,Português (Brasil),NaN,https://www.gov.br/anvisa/pt-br
4,https://www.gov.br/anvisa/pt-br/assuntos/notic...,200.0,0.657,467631.0,7128.0,114.0,2.0,True,Anvisa aprova regularização de cosméticos arte...,NaN,https://www.gov.br/anvisa/pt-br
5,https://www.gov.br/anvisa/pt-br/assuntos/notic...,200.0,0.877,464085.0,4496.0,107.0,2.0,True,CMED divulga 9° edição do Anuário Estatístico ...,NaN,https://www.gov.br/anvisa/pt-br
6,https://www.gov.br/imprensanacional/,200.0,0.833,324748.0,2430.0,69.0,10.0,True,Imprensa Nacional,NaN,https://www.gov.br/imprensanacional/pt-br
7,https://www.gov.br/imprensanacional/pt-br/cana...,200.0,0.599,274183.0,4413.0,68.0,2.0,True,O legado de Lêda Watson na Imprensa Nacional,NaN,https://www.gov.br/imprensanacional/pt-br
8,https://www.gov.br/imprensanacional/pt-br/cana...,200.0,0.632,260449.0,3304.0,68.0,2.0,True,Imprensa Nacional renova parque gráfico com im...,NaN,https://www.gov.br/imprensanacional/pt-br
9,https://www25.senado.leg.br/web/atividade/plen...,200.0,1.026,29742.0,1071.0,70.0,2.0,False,Plenário - Senado Federal,NaN,https://www25.senado.leg.br/web/atividade


In [72]:
navigation_summary = (
    df_navigation
    .groupby("source_url")
    .agg(
        pages_tested=("internal_url", "count"),

        status_min=("status", "min"),
        status_max=("status", "max"),

        avg_response_time=(
            "response_time",
            "mean"
        ),

        min_text_length=(
            "text_length",
            "min"
        ),

        max_text_length=(
            "text_length",
            "max"
        ),

        captcha_reference=(
            "captcha_reference",
            "any"
        )
    )
    .reset_index()
)

In [73]:
navigation_summary["navigation_ok"] = (
    (navigation_summary["pages_tested"] > 0)
    &
    (navigation_summary["status_min"] == 200)
    &
    (navigation_summary["status_max"] == 200)
)

In [74]:
navigation_summary[
    [
        "source_url",
        "pages_tested",
        "status_min",
        "status_max",
        "navigation_ok",
        "captcha_reference",
        "avg_response_time"
    ]
]

,source_url,pages_tested,status_min,status_max,navigation_ok,captcha_reference,avg_response_time
0,https://deirn.sdoe.com.br,0,403.0,403.0,False,False,NaN
1,https://legislativo.camaraparnamirim.rn.gov.br/,3,200.0,200.0,True,True,1.301667
2,https://www.al.rn.leg.br,3,200.0,200.0,True,True,0.596667
3,https://www.camara.leg.br/busca-portal/proposi...,3,200.0,200.0,True,False,0.803333
4,https://www.cmnat.rn.gov.br,3,200.0,200.0,True,True,0.480000
5,https://www.confaz.fazenda.gov.br,0,NaN,NaN,False,False,NaN
6,https://www.gov.br/anvisa,3,200.0,200.0,True,True,0.823333
7,https://www.gov.br/anvisa/pt-br,3,200.0,200.0,True,True,0.739000
8,https://www.gov.br/cade,0,NaN,NaN,False,False,NaN
9,https://www.gov.br/conitec,3,200.0,200.0,True,True,0.817000


## Analise de tentaiva dos resultados NaN

In [75]:
df_navigation[
    df_navigation["source_url"].isin([
        "https://www.confaz.fazenda.gov.br",
        "https://www.gov.br/cade",
        "https://www.gov.br/senacon"
    ])
][
    [
        "source_url",
        "internal_url",
        "status",
        "error"
    ]
]

,source_url,internal_url,status,error
34,https://www.confaz.fazenda.gov.br,NaN,NaN,HTTPSConnectionPool(host='www.confaz.fazenda.g...
35,https://www.gov.br/cade,NaN,NaN,"HTTPSConnectionPool(host='www.gov.br', port=44..."
36,https://www.gov.br/senacon,NaN,NaN,"HTTPSConnectionPool(host='www.gov.br', port=44..."


In [76]:
urls_investigar = [
    "https://www.confaz.fazenda.gov.br",
    "https://www.gov.br/cade",
    "https://www.gov.br/senacon"
]

for url in urls_investigar:

    response = requests.get(
        url,
        timeout=15
    )

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    print("=" * 70)
    print(url)
    print("Status:", response.status_code)
    print("URL final:", response.url)
    print("HTML:", len(response.content))
    print("Links:", len(soup.find_all("a")))
    print("Título:", soup.title.get_text(" ", strip=True) if soup.title else None)

https://www.confaz.fazenda.gov.br
Status: 200
URL final: https://www.confaz.fazenda.gov.br/
HTML: 105979
Links: 187
Título: Sítio CONFAZ — Conselho Nacional de Política Fazendária CONFAZ
https://www.gov.br/cade
Status: 200
URL final: https://www.gov.br/cade/pt-br
HTML: 200215
Links: 499
Título: Página Inicial — Conselho Administrativo de Defesa Econômica
https://www.gov.br/senacon
Status: 200
URL final: https://www.gov.br/mj/pt-br/assuntos/seus-direitos/consumidor
HTML: 186394
Links: 500
Título: Senacon — Ministério da Justiça e Segurança Pública


In [77]:
for url in [
    "https://www.confaz.fazenda.gov.br",
    "https://www.gov.br/cade",
    "https://www.gov.br/senacon"
]:
    row = df_navigation[
        df_navigation["source_url"] == url
    ]

    print("=" * 70)
    print(url)

    for error in row["error"].dropna():
        print(error)

https://www.confaz.fazenda.gov.br
HTTPSConnectionPool(host='www.confaz.fazenda.gov.br', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("HTTPSConnection(host='www.confaz.fazenda.gov.br', port=443): Failed to resolve 'www.confaz.fazenda.gov.br' ([Errno 11001] getaddrinfo failed)"))
https://www.gov.br/cade
HTTPSConnectionPool(host='www.gov.br', port=443): Max retries exceeded with url: /cade (Caused by NameResolutionError("HTTPSConnection(host='www.gov.br', port=443): Failed to resolve 'www.gov.br' ([Errno 11001] getaddrinfo failed)"))
https://www.gov.br/senacon
HTTPSConnectionPool(host='www.gov.br', port=443): Max retries exceeded with url: /senacon (Caused by NameResolutionError("HTTPSConnection(host='www.gov.br', port=443): Failed to resolve 'www.gov.br' ([Errno 11001] getaddrinfo failed)"))


## Recomendações

Com base nos testes realizados:

- Fontes que retornam os dados diretamente no HTML podem utilizar `requests` e `BeautifulSoup`;
- Fontes cujo conteúdo é carregado dinamicamente devem ter suas requisições de rede analisadas para identificar possíveis APIs ou endpoints;
- Selenium ou Playwright devem ser considerados apenas quando a extração depender efetivamente da execução de JavaScript;
- Os seletores utilizados devem ser mantidos isolados para facilitar futuras atualizações caso a estrutura HTML das fontes seja alterada.

## Conclusão

Os testes demonstraram que, **nas condições e páginas analisadas**, as fontes válidas apresentaram viabilidade para coleta automatizada através de requisições HTTP e parsing de HTML.

Foi possível:

- Acessar as fontes;
- Receber e interpretar o HTML;
- Extrair textos, headings, parágrafos e links;
- Repetir requisições sem observar bloqueio progressivo;
- Acessar páginas internas;
- Diferenciar referências a CAPTCHA de bloqueios efetivos;
- Identificar e diferenciar falhas temporárias de DNS de bloqueios do servidor.

Não foi identificado `HTTP 429` ou outro comportamento que demonstrasse **rate limiting** durante os testes realizados.

O caso do **DEI/RN permanece pendente**, pois a URL utilizada não corresponde ao endereço válido da fonte pretendida.

Também foram identificadas falhas temporárias de resolução DNS em uma das execuções. Isso demonstra a importância de diferenciar:

```text
Falha de DNS/conectividade
        ≠
Resposta HTTP de erro
        ≠
Bloqueio anti-bot
```

De modo geral, os testes realizados indicam que **as fontes válidas analisadas são tecnicamente acessíveis para a continuidade do desenvolvimento do scraping**.

Estabilidade entre execuções: Nas baterias iniciais, as fontes acessíveis mantiveram predominantemente respostas HTTP 200 e permitiram extração e navegação. Em uma execução posterior, CONFAZ, CADE e Senacon apresentaram ausência de páginas testadas e valores NaN para status HTTP. A análise dos erros brutos identificou NameResolutionError, com mensagens Failed to resolve e getaddrinfo failed, caracterizando falha de resolução DNS antes do estabelecimento da conexão HTTP. Portanto, não houve resposta HTTP dos servidores nessas tentativas e os valores NaN representam ausência de resposta, não códigos retornados pelas fontes. Em nova tentativa realizada posteriormente, as três fontes voltaram a responder HTTP 200 e forneceram HTML normalmente. Dessa forma, a ocorrência foi classificada como falha transitória de conectividade/resolução DNS, e não como evidência de bloqueio anti-scraping.